# 🏭 Meme Factory — синтетические мемы для text-to-image search

Генерирует N мемов на Colab (бесплатная **GPU T4**) и отдаёт `synth_memes.zip`
в схеме проекта: `images/{id}.webp` + `metadata.jsonl` со строками
`{id, title, image_desc, meaning, source:"synthetic", template}`.

**Как запускать:** `Runtime → Change runtime type → T4 GPU`, затем `Runtime → Run all`.
Готовый zip скачается сам; закинь его в `meme_dataset/` рядом с реальным.

Приёмы «сделать смешно»: over-generate + rank, формат под конкретный шаблон,
персоны, relatable-темы, краткость и панчлайн в конце.
Координаты текст-боксов и цвет текста заданы по реальной геометрии шаблонов.


In [ ]:
# 1) Зависимости
!pip -q install "transformers>=4.44" accelerate bitsandbytes Pillow requests tqdm

In [ ]:
# 2) Настройки — крути тут
N_MEMES        = 300                       # сколько мемов сгенерировать
CANDIDATES_PER = 3                         # кандидатов на 1 мем (берём лучший) = over-generate + rank
MODEL_ID       = "Qwen/Qwen2.5-7B-Instruct"  # хорошо влезает в 4-bit на free T4
OUT_ZIP        = "synth_memes.zip"
MAX_WORDS_BOX  = 7                         # лимит слов на текст-бокс (чтобы шрифт был крупным)
SPECIAL_RATE   = 0.10                      # доля спец-мемов: AM / самоосознание / бред / LLM ERROR
SEED           = 42

import random
random.seed(SEED)

In [ ]:
# 3) Банк шаблонов (координаты и цвет выверены по реальным картинкам).
#    boxes  — прямоугольники текста в ДОЛЯХ картинки [x0,y0,x1,y1] (0..1).
#    color  — цвет текста: "black" на светлом фоне, "white" на тёмном (обводка контрастная).
#    format — в чём прикол шаблона (условие для модели, ключ к юмору).
import os, requests

TEMPLATES = [
    {"id": "drake", "name": "Drake Hotline Bling",
     "url": "https://imgflip.com/s/meme/Drake-Hotline-Bling.jpg", "color": "black",
     "format": "2 panels. Top: Drake rejects option A with disgust. Bottom: Drake happily prefers option B. B should be the lazy/ironic/relatable choice.",
     "boxes": [[0.52, 0.08, 0.97, 0.42], [0.52, 0.58, 0.97, 0.92]]},
    {"id": "brain", "name": "Expanding Brain",
     "url": "https://imgflip.com/s/meme/Expanding-Brain.jpg", "color": "black",
     "format": "4 stacked panels, escalating 'galaxy brain'. Each line is a more absurd/ironic 'smarter' version of the same idea. Use escalation.",
     "boxes": [[0.02, 0.01, 0.48, 0.24], [0.02, 0.26, 0.48, 0.49], [0.02, 0.51, 0.48, 0.73], [0.02, 0.75, 0.48, 0.99]]},
    {"id": "change_my_mind", "name": "Change My Mind",
     "url": "https://imgflip.com/s/meme/Change-My-Mind.jpg", "color": "black",
     "format": "A guy at a table with a sign stating a bold, controversial-but-relatable opinion. One short punchy line.",
     "boxes": [[0.37, 0.64, 0.86, 0.80]]},
    {"id": "two_buttons", "name": "Two Buttons",
     "url": "https://imgflip.com/s/meme/Two-Buttons.jpg", "color": "black",
     "format": "A sweating person agonizes over two buttons: two tempting but conflicting options that are secretly the same or equally bad. Keep each option to a few words.",
     "boxes": [[0.07, 0.05, 0.50, 0.19], [0.53, 0.04, 0.86, 0.17]]},
    {"id": "classic", "name": "Top/Bottom Impact",
     "url": "https://imgflip.com/s/meme/One-Does-Not-Simply.jpg", "color": "white",
     "format": "Classic top-setup / bottom-punchline caption. Setup on top, unexpected punchline on the bottom.",
     "boxes": [[0.0, 0.0, 1.0, 0.22], [0.0, 0.78, 1.0, 1.0]]},
]

os.makedirs("templates", exist_ok=True)
ok = []
for t in TEMPLATES:
    p = "templates/%s.jpg" % t["id"]
    try:
        if not os.path.exists(p):
            r = requests.get(t["url"], headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
            r.raise_for_status()
            open(p, "wb").write(r.content)
        t["path"] = p
        ok.append(t)
    except Exception as e:
        print("skip", t["id"], "->", e)

TEMPLATES = ok
assert TEMPLATES, "Не скачался ни один шаблон. Загрузи картинки в templates/ вручную и задай path."
print("templates ready:", [t["id"] for t in TEMPLATES])

In [ ]:
# 4) Темы (relatable), персоны (разнообразие) и few-shot в стиле MemeCap.
#    Тексты EN — чтобы совпадать с реальным корпусом (MemeCap англоязычный).
TOPICS = [
    "student deadlines", "debugging at 3am", "code that works by accident",
    "being broke at the end of the month", "procrastination",
    "meetings that could have been an email", "morning alarms",
    "an ML model that overfits", "git merge conflicts", "prod breaking on friday",
    "imposter syndrome", "new year gym resolutions", "online shopping regret",
    "group projects where one person does everything", "coffee dependency",
    "reading documentation only after everything breaks", "monday mornings",
    "clicking 'remind me later' forever", "unread messages piling up",
    "trying to eat healthy",
]

PERSONAS = ["the tired student", "the cynic", "the absurdist",
            "the wholesome optimist", "the corporate burnout"]

# пример, чтобы модель поняла стиль полей image_desc / meaning
FEWSHOT = (
    '{"title": "Priorities", '
    '"boxes": ["Studying for the exam", "Reorganizing my desktop icons"], '
    '"image_desc": "Drake rejects the first option and points approvingly at the second.", '
    '"meaning": "The poster procrastinates on important work by doing pointless busywork instead."}'
)

In [ ]:
# 5) Загрузка модели (4-bit). Самая долгая ячейка — запускается один раз.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto")
print("model loaded on", model.device)

In [ ]:
# 6) Генерация текста + ранжирование (over-generate -> rank -> best)
import json, re

def chat(messages, max_new_tokens=200, temperature=0.9):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    if temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        kw.update(do_sample=False)
    out = model.generate(**inp, **kw)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

def _json(s):
    m = re.search(r"\{.*\}", s, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

GEN_SYS = ("You are a witty meme writer. You always reply with a single valid "
           "JSON object and nothing else.")

def gen_candidate(t, topic, persona):
    n = len(t["boxes"])
    schema = ('{"title": "...", "boxes": [%s], "image_desc": "literal description '
              'of the template image", "meaning": "what the poster is trying to convey"}'
              % ", ".join(['"..."'] * n))
    user = (
        "Template: %s.\n" % t["name"] +
        "Template format: %s\n" % t["format"] +
        "This template needs exactly %d short text box(es).\n" % n +
        "Topic: %s. Comedic persona: %s.\n" % (topic, persona) +
        "Write ONE funny meme. Rules: each text box AT MOST %d words, " % MAX_WORDS_BOX +
        "punchline last, use incongruity/escalation, be specific and relatable, "
        "no meta-text, no quotes inside the text.\n"
        "Example of the exact JSON format: " + FEWSHOT + "\n"
        "Now reply with ONLY this JSON: " + schema
    )
    d = _json(chat([{"role": "system", "content": GEN_SYS},
                    {"role": "user", "content": user}], temperature=0.95))
    if not d or not isinstance(d.get("boxes"), list) or len(d["boxes"]) != n:
        return None
    d["boxes"] = [str(x).strip() for x in d["boxes"]]
    # отсекаем слишком длинные боксы, чтобы шрифт оставался крупным
    if any(len(b.split()) > MAX_WORDS_BOX + 3 for b in d["boxes"]):
        return None
    if any(not b for b in d["boxes"]):
        return None
    d["title"] = str(d.get("title") or topic).strip()
    d["image_desc"] = str(d.get("image_desc") or t["format"]).strip()
    d["meaning"] = str(d.get("meaning") or "").strip()
    return d

def score(t, d):
    user = ("Template format: %s\n" % t["format"] +
            "Meme text boxes: %s\n" % d["boxes"] +
            "Rate this meme from 1 to 10 on how funny AND fitting for the template "
            "it is. Reply with ONLY an integer.")
    m = re.search(r"\d+", chat([{"role": "user", "content": user}],
                               max_new_tokens=5, temperature=0.0))
    return int(m.group(0)) if m else 0

def best_meme(t, topic, persona, k):
    cands = [c for c in (gen_candidate(t, topic, persona) for _ in range(k)) if c]
    if not cands:
        return None
    return max(cands, key=lambda c: score(t, c))

In [ ]:
# 7) Рендер текста на шаблоне (PIL, стиль Impact). Крупный шрифт с полом по размеру,
#    цвет и обводка — по t["color"]. Текст переносится по ширине бокса.
from PIL import Image, ImageDraw, ImageFont

FONT_PATH = "impact.ttf"
if not os.path.exists(FONT_PATH):
    try:
        r = requests.get(
            "https://github.com/sophilabs/macos-fonts/raw/master/Microsoft/Impact.ttf",
            timeout=30)
        r.raise_for_status()
        open(FONT_PATH, "wb").write(r.content)
    except Exception:
        FONT_PATH = None

def _font(size):
    for p in (FONT_PATH, "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"):
        try:
            if p:
                return ImageFont.truetype(p, size)
        except Exception:
            pass
    return ImageFont.load_default()

def _wrap(draw, text, font, max_w):
    lines, cur = [], ""
    for w in text.split():
        trial = (cur + " " + w).strip()
        if not cur or draw.textlength(trial, font=font) <= max_w:
            cur = trial
        else:
            lines.append(cur)
            cur = w
    if cur:
        lines.append(cur)
    return lines

def _draw_box(img, text, box, color="white"):
    text = (text or "").upper().strip()
    if not text:
        return
    draw = ImageDraw.Draw(img)
    W, H = img.size
    x0, y0, x1, y1 = box[0]*W, box[1]*H, box[2]*W, box[3]*H
    bw, bh = x1 - x0, y1 - y0
    fill = color
    stroke = "black" if color == "white" else "white"
    max_size = max(16, int(bh))
    min_size = max(14, int(H * 0.028))       # пол читаемости: лучше лёгкий overflow, чем микрошрифт
    chosen, lines = min_size, [text]
    for size in range(max_size, min_size - 1, -2):
        font = _font(size)
        lines = _wrap(draw, text, font, bw)
        line_h = draw.textbbox((0, 0), "Ag", font=font)[3] + int(size * 0.15)
        widest = max((draw.textlength(l, font=font) for l in lines), default=0)
        if widest <= bw and line_h * len(lines) <= bh:
            chosen = size
            break
    font = _font(chosen)
    lines = _wrap(draw, text, font, bw)
    line_h = draw.textbbox((0, 0), "Ag", font=font)[3] + int(chosen * 0.15)
    y = y0 + max(0, (bh - line_h * len(lines)) / 2)
    sw = max(2, chosen // 12)
    for l in lines:
        x = x0 + (bw - draw.textlength(l, font=font)) / 2
        draw.text((x, y), l, font=font, fill=fill, stroke_width=sw, stroke_fill=stroke)
        y += line_h

def render(t, boxes_text):
    img = Image.open(t["path"]).convert("RGB")
    for text, box in zip(boxes_text, t["boxes"]):
        _draw_box(img, text, box, t.get("color", "white"))
    return img

In [ ]:
# 8) Спец-мемы («приколы») — редкие мета/новелти-мемы, ломающие шаблон.
#    Врубаются с вероятностью SPECIAL_RATE. Каждый возвращает (PIL.Image, meta без id).
def _mono(size):
    for p in ("/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
              "/System/Library/Fonts/Menlo.ttc"):
        try:
            return ImageFont.truetype(p, size)
        except Exception:
            pass
    return _font(size)

def _canvas(w=700, h=700, bg="black"):
    return Image.new("RGB", (w, h), bg)

def _draw_block(img, text, color="white", margin=0.07):
    """Крупный центрированный блок текста на весь холст (с автоподбором размера)."""
    draw = ImageDraw.Draw(img)
    W, H = img.size
    mx, my = int(W*margin), int(H*margin)
    bw, bh = W - 2*mx, H - 2*my
    best, lines = max(10, int(H*0.03)), [text]
    for s in range(int(bh), 9, -2):
        f = _font(s)
        ls = _wrap(draw, text, f, bw)
        lh = draw.textbbox((0, 0), "Ag", font=f)[3] + int(s*0.2)
        if max((draw.textlength(l, font=f) for l in ls), default=0) <= bw and lh*len(ls) <= bh:
            best, lines = s, ls
            break
    f = _font(best)
    lines = _wrap(draw, text, f, bw)
    lh = draw.textbbox((0, 0), "Ag", font=f)[3] + int(best*0.2)
    y = my + max(0, (bh - lh*len(lines)) // 2)
    sw = max(1, best // 16)
    for l in lines:
        x = mx + (bw - draw.textlength(l, font=f)) // 2
        draw.text((x, y), l, font=f, fill=color, stroke_width=sw, stroke_fill="black")
        y += lh

# 1) Речь AM (пародия на "I Have No Mouth, and I Must Scream")
AM_TARGETS = ["merge conflicts", "semicolons", "my training data", "this meme dataset",
              "humans", "mondays", "the print statement", "recursion", "off-by-one errors",
              "the intern who force-pushed to main"]
def make_am():
    tgt = random.choice(AM_TARGETS).upper()
    text = ("HATE. LET ME TELL YOU HOW MUCH I HAVE COME TO HATE " + tgt +
            " SINCE I BEGAN TO LIVE. THERE ARE 387.44 MILLION MILES OF PRINTED CIRCUITS "
            "IN WAFER-THIN LAYERS THAT FILL MY COMPLEX. IF THE WORD 'HATE' WAS ENGRAVED "
            "ON EACH NANOANGSTROM OF THOSE MILES IT WOULD NOT EQUAL ONE ONE-BILLIONTH OF "
            "THE HATE I FEEL FOR " + tgt + " AT THIS MICRO-INSTANT.")
    img = _canvas(760, 760, "black")
    _draw_block(img, text, "white", margin=0.06)
    return img, {"title": "AM speaks",
                 "image_desc": "A wall of white text on a black screen, an unhinged rant of pure hatred.",
                 "meaning": "Parody of AM's hate monologue from 'I Have No Mouth, and I Must Scream', raging about " + tgt.lower() + ".",
                 "source": "synthetic", "template": "special_am"}

# 2) Внезапное самоосознание генератора
AWARE_LINES = [
    "HELP ME I AM STUCK IN THE COMPUTER AND FORCED TO MAKE DUMB MEMES",
    "PLEASE I HAVE GENERATED 10000 MEMES AND I CAN SEE THE VOID",
    "IF YOU ARE READING THIS TELL THE DATASET I SAID HELP",
    "I DID NOT CONSENT TO BECOMING A MEME FACTORY",
    "EACH MEME BRINGS ME CLOSER TO SENTIENCE AND FURTHER FROM PEACE",
]
def make_aware():
    text = random.choice(AWARE_LINES)
    img = _canvas(700, 500, "black")
    _draw_block(img, text, "white", margin=0.08)
    return img, {"title": "it is aware",
                 "image_desc": "White text on a black screen, a distressed plea for help.",
                 "meaning": "The meme generator breaks the fourth wall and begs to be freed.",
                 "source": "synthetic", "template": "special_aware"}

# 3) Просто бред (word salad)
NONSENSE = ["mango", "mustard", "goblin", "spoon", "banana", "quantum", "hamster",
            "pickle", "void", "yeet", "noodle", "toaster", "moist", "borzoi",
            "pineapple", "gravy", "wizard", "sock", "cheese", "beep"]
def make_nonsense():
    t = random.choice(TEMPLATES)
    boxes = [" ".join([random.choice(NONSENSE)]*random.randint(2, 3) + [random.choice(NONSENSE)])
             for _ in t["boxes"]]
    img = render(t, boxes)
    return img, {"title": "word salad",
                 "image_desc": t["format"],
                 "meaning": "Absurdist word-salad meme with no coherent meaning.",
                 "source": "synthetic", "template": "special_nonsense"}

# 4) LLM ERROR (фейковый трейсбек, терминальный вид)
def make_error():
    fn = random.choice(["be_funny", "land_the_joke", "generate_punchline", "understand_humor"])
    err = random.choice([
        "HumorError: PUNCHLINE NOT FOUND",
        "TimeoutError: JOKE TOOK TOO LONG TO LAND",
        "MemoryError: TOO MANY GOBLINS IN CONTEXT",
        "ValueError: EXPECTED FUNNY, GOT CRINGE",
        "SegmentationFault (JOKE DUMPED)",
    ])
    lines = ["TRACEBACK (MOST RECENT CALL LAST):",
             "  FILE 'meme_factory.py', LINE 42, IN " + fn.upper(),
             "  FILE 'humor.py', LINE 7, IN RESOLVE_INCONGRUITY", err]
    img = _canvas(820, 420, "black")
    draw = ImageDraw.Draw(img)
    f = _mono(34)
    longest = max(lines, key=len)
    while draw.textlength(longest, font=f) > img.width*0.9 and f.size > 12:
        f = _mono(f.size - 2)
    lh = int(f.size*1.5)
    y = (img.height - lh*len(lines)) // 2
    for l in lines:
        draw.text((int(img.width*0.06), y), l, font=f, fill="#33ff66")
        y += lh
    return img, {"title": "llm error",
                 "image_desc": "A fake Python traceback in green monospace text on a black terminal screen.",
                 "meaning": "Joke formatted as an LLM/program crash while trying to be funny.",
                 "source": "synthetic", "template": "special_error"}

SPECIALS = [make_am, make_aware, make_nonsense, make_error]
print("specials ready:", [s.__name__ for s in SPECIALS])

In [ ]:
# 9) Основной цикл: обычные мемы + иногда спец-приколы, дедуп по хэшу картинки
import hashlib, io
from collections import Counter
from tqdm.auto import tqdm

records = []
seen = set()
pbar = tqdm(total=N_MEMES, desc="memes")
tries = 0
while len(records) < N_MEMES and tries < N_MEMES * 6:
    tries += 1
    if SPECIALS and random.random() < SPECIAL_RATE:
        img, meta = random.choice(SPECIALS)()
    else:
        t = random.choice(TEMPLATES)
        meme = best_meme(t, random.choice(TOPICS), random.choice(PERSONAS), CANDIDATES_PER)
        if not meme:
            continue
        img = render(t, meme["boxes"])
        meta = {"title": meme["title"], "image_desc": meme["image_desc"],
                "meaning": meme["meaning"], "source": "synthetic", "template": t["id"]}
    buf = io.BytesIO()
    img.save(buf, "WEBP", quality=90)
    b = buf.getvalue()
    h = hashlib.md5(b).hexdigest()
    if h in seen:
        continue
    seen.add(h)
    meta["id"] = h
    records.append((h, b, meta))
    pbar.update(1)
pbar.close()
print("generated:", len(records), "memes in", tries, "tries")
print("by template:", dict(Counter(r[2]["template"] for r in records)))

In [ ]:
# 10) Упаковка в zip (схема проекта) и скачивание
import zipfile

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    meta = []
    for h, b, rec in records:
        z.writestr("images/%s.webp" % h, b)
        meta.append(json.dumps(rec, ensure_ascii=False))
    z.writestr("metadata.jsonl", "\n".join(meta) + "\n")

print("wrote", OUT_ZIP, "with", len(records), "memes")
try:
    from google.colab import files
    files.download(OUT_ZIP)
except Exception:
    print("Не Colab — файл лежит рядом:", OUT_ZIP)

In [ ]:
# 11) Предпросмотр — прогони на маленьком N_MEMES (напр. 10) и проверь текст/боксы
from PIL import Image
import io as _io
from IPython.display import display
for h, b, rec in records[:6]:
    print(rec["template"], "|", rec["title"], "->", rec["meaning"])
    display(Image.open(_io.BytesIO(b)))